# QUN Kaggle GPU Worker — clean MVP

Проверенный маршрут: **QUN Studio → очередь QUN Compute → ComfyUI на GPU Kaggle → QUN Library**.

## Перед запуском
1. В настройках Kaggle включи **GPU**.
2. В Kaggle Secrets добавь `QUN_SUPABASE_PUBLISHABLE_KEY` (значение — publishable key проекта Supabase; не service-role key).
3. Не публикуй вывод ячейки регистрации и не сохраняй токены в GitHub.
4. Выполняй ячейки сверху вниз. Последняя ячейка остаётся работать, пока узел принимает задания.

Для первого теста используй английский prompt: текущая модель DreamShaper обычно лучше следует подробным английским описаниям. Автоперевод русского prompt пока не реализован.

In [ ]:
# 1. Проверка GPU и подготовка исходников
from pathlib import Path
import os, sys, subprocess, time, json, urllib.request

import torch
if not torch.cuda.is_available():
    raise RuntimeError('В Kaggle включи GPU Accelerator и перезапусти сессию.')
print('GPU:', torch.cuda.get_device_name(0))

QUN = Path('/kaggle/working/QUN')
COMFY = Path('/kaggle/working/ComfyUI')
if not QUN.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Serega40in/QUN.git',str(QUN)],check=True)
if not COMFY.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(COMFY/'requirements.txt')],check=True)
STUDIO = QUN / 'docs' / 'studio'
WORKER = STUDIO / 'comfyui_worker.py'
assert WORKER.is_file(), f'Worker не найден: {WORKER}'
print('QUN и ComfyUI готовы.')

In [ ]:
# 2. Загрузка модели DreamShaper 8
from huggingface_hub import hf_hub_download
MODEL_DIR = COMFY / 'models' / 'checkpoints'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL = MODEL_DIR / 'dreamshaper_8.safetensors'
if not MODEL.is_file() or MODEL.stat().st_size < 1_000_000_000:
    downloaded = hf_hub_download(repo_id='digiplay/DreamShaper_8', filename='dreamshaper_8.safetensors', local_dir=str(MODEL_DIR))
    MODEL = Path(downloaded)
if not MODEL.is_file() or MODEL.stat().st_size < 1_000_000_000:
    raise RuntimeError('Модель не загрузилась полностью. Проверь доступность репозитория модели.')
print('Model ready:', MODEL.name, '| GiB:', round(MODEL.stat().st_size / 1024**3, 2))

In [ ]:
# 3. Фиксированный API workflow. Задание не может передавать произвольный workflow или путь к файлу.
WORKFLOW = COMFY / 'qun_image_api_workflow.json'
workflow = {
 '1': {'class_type':'CheckpointLoaderSimple','inputs':{'ckpt_name':MODEL.name}},
 '2': {'class_type':'CLIPTextEncode','inputs':{'text':'{{PROMPT}}','clip':['1',1]}},
 '3': {'class_type':'CLIPTextEncode','inputs':{'text':'low quality, blurry, low detail, text, watermark, logo, malformed anatomy, extra fingers, duplicate subjects','clip':['1',1]}},
 '4': {'class_type':'EmptyLatentImage','inputs':{'width':512,'height':512,'batch_size':1}},
 '5': {'class_type':'KSampler','inputs':{'seed':4061109,'steps':24,'cfg':7.0,'sampler_name':'euler','scheduler':'normal','denoise':1.0,'model':['1',0],'positive':['2',0],'negative':['3',0],'latent_image':['4',0]}},
 '6': {'class_type':'VAEDecode','inputs':{'samples':['5',0],'vae':['1',2]}},
 '7': {'class_type':'SaveImage','inputs':{'filename_prefix':'{{OUTPUT_PREFIX}}','images':['6',0]}}
}
WORKFLOW.write_text(json.dumps(workflow,ensure_ascii=False,indent=2),encoding='utf-8')
print('Workflow saved:', WORKFLOW)

In [ ]:
# 4. Безопасная регистрация узла QUN
from kaggle_secrets import UserSecretsClient
from getpass import getpass
import importlib.util

secrets = UserSecretsClient()
try:
    SUPABASE_KEY = secrets.get_secret('QUN_SUPABASE_PUBLISHABLE_KEY')
except Exception as exc:
    raise RuntimeError('Добавь QUN_SUPABASE_PUBLISHABLE_KEY в Kaggle → Add-ons → Secrets.') from exc
if not SUPABASE_KEY:
    raise RuntimeError('Kaggle Secret QUN_SUPABASE_PUBLISHABLE_KEY пуст.')
SUPABASE_URL = 'https://qdxjltqlnphvzafapncm.supabase.co'
REGISTRY = Path('/kaggle/working/.config/qun/render-node.json')
REGISTRY.parent.mkdir(parents=True,exist_ok=True)
os.environ.update({'QUN_SUPABASE_URL':SUPABASE_URL,'QUN_SUPABASE_PUBLISHABLE_KEY':SUPABASE_KEY,'QUN_NODE_PLATFORM':'linux-gpu','QUN_NODE_REGISTRY_FILE':str(REGISTRY)})

node_id = None
node_token = None
try:
    node_id = secrets.get_secret('QUN_NODE_ID')
    node_token = secrets.get_secret('QUN_NODE_TOKEN')
except Exception:
    pass
if node_id and node_token:
    REGISTRY.write_text(json.dumps({'nodeId':node_id,'nodeToken':node_token,'platform':'linux-gpu'},indent=2)+'\\n',encoding='utf-8')
else:
    sys.path.insert(0,str(STUDIO))
    import register_gpu_worker as registry_api
    code = getpass('Вставь одноразовый код из QUN Studio → Подключить GPU-узел: ').strip()
    if len(code) < 20:
        raise RuntimeError('Код слишком короткий. Создай новый одноразовый код в QUN Studio.')
    data = registry_api.request_registration('QUN Kaggle GPU',code)
    registry_api.save_credentials(data,REGISTRY)
    print('Узел зарегистрирован. Node ID:',data['nodeId'])
    print('Токен не выводится; он сохранён только в локальном файле Kaggle.')
print('Registry file exists:',REGISTRY.is_file())
print('Node token present:',bool(json.loads(REGISTRY.read_text(encoding='utf-8')).get('nodeToken')))

In [ ]:
# 5. Запуск ComfyUI локально
COMFY_LOG = open('/kaggle/working/comfyui.log','w',encoding='utf-8')
comfy_process = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188'],cwd=str(COMFY),stdout=COMFY_LOG,stderr=subprocess.STDOUT,env=os.environ.copy())
COMFY_URL = 'http://127.0.0.1:8188'
for attempt in range(60):
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI завершился. Проверь /kaggle/working/comfyui.log.')
    try:
        with urllib.request.urlopen(COMFY_URL+'/system_stats',timeout=5) as response:
            print('ComfyUI ready:',response.status,COMFY_URL)
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError('ComfyUI не ответил за 3 минуты. Проверь лог.')

In [ ]:
# 6. Запуск рабочего узла QUN — оставь ячейку выполняться
os.environ.update({
 'QUN_COMFYUI_URL':COMFY_URL,
 'QUN_COMFYUI_WORKFLOW_IMAGE':str(WORKFLOW),
 'QUN_COMFYUI_OUTPUT_DIR':str(COMFY/'output'),
 'QUN_NODE_REGISTRY_FILE':str(REGISTRY),
 'QUN_POLL_SECONDS':'8',
 'QUN_JOB_TIMEOUT_SECONDS':'900',
})
os.chdir(STUDIO)
print('QUN GPU Worker starting. В QUN Studio сначала проверь очередь, затем создай задание image_generation.')
print('Остановка: Interrupt у этой ячейки; затем останови ComfyUI при необходимости.')
subprocess.run([sys.executable,'-u',str(WORKER)],cwd=str(STUDIO),env=os.environ.copy(),check=False)

## Проверка результата

1. Дождись heartbeat узла в QUN Studio.
2. Запусти `health_check` / «Проверка очереди».
3. Создай задание «Генерация изображения» с подробным prompt на английском.
4. Проверь статус `succeeded` и наличие PNG в QUN Library.

**Безопасность:** не сохраняй в публичных выводах publishable key, одноразовый код или `nodeToken`. Для повторных сессий можно добавить `QUN_NODE_ID` и `QUN_NODE_TOKEN` в Kaggle Secrets; иначе регистрируй новый узел одноразовым кодом. Видео и аудио на этом MVP не настроены.